# Validation analysis

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# Validation/Test Analysis for Coastal Transformer Model

This notebook loads inference outputs from `evaluate.py`, computes summary metrics,
builds scatter diagnostics for `Hs` and `Tp`, and creates Taylor diagrams using
the `metocean-stats` library for validation vs unseen test comparison.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

ROOT = Path("..") if (Path.cwd().name == "notebooks") else Path(".")
cfg_path = ROOT / "configs" / "training.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) if cfg_path.exists() else {}

# Optional override: set to a run folder name (e.g. 'LSTM_newstatic_test1') or full path.
RUN_FOLDER_OVERRIDE = "./results/final_lstm_nostatic"  # Example: 'outerfjord_58sites_test1' or '/full/path/to/results/outerfjord_58sites_test1'


# RUN_FOLDER_OVERRIDE = None  # Set to None to disable override
def _resolve_results_dir(root: Path, config: dict, run_override: str | None = None):
    log_cfg = config.get("logging", {}) or {}
    configured_out = Path(log_cfg.get("output_dir", "results/coastal_transformer"))
    configured_dir = configured_out if configured_out.is_absolute() else (root / configured_out)

    if run_override:
        override_path = Path(run_override)
        if override_path.is_absolute():
            candidate = override_path
        elif override_path.parent != Path("."):
            candidate = root / override_path
        else:
            candidate = root / "results" / run_override
        return candidate, "override"

    # Primary source: the run folder configured for training/evaluation.
    if configured_dir.exists():
        return configured_dir, "logging.output_dir"

    # Fallback only when configured directory is missing.
    results_root = root / "results"
    if results_root.exists():
        candidates = []
        for d in results_root.rglob("*"):
            if not d.is_dir():
                continue
            val_csv = d / "predictions_val.csv"
            test_csv = d / "predictions_test.csv"
            if val_csv.exists() and test_csv.exists():
                mtime = max(val_csv.stat().st_mtime, test_csv.stat().st_mtime)
                candidates.append((mtime, d))
        if candidates:
            candidates.sort(key=lambda x: x[0], reverse=True)
            return candidates[0][1], "latest-results-folder"

    return configured_dir, "logging.output_dir (directory missing)"


RESULTS_DIR, RESULTS_SOURCE = _resolve_results_dir(ROOT, cfg, RUN_FOLDER_OVERRIDE)
VAL_CSV = RESULTS_DIR / "predictions_val.csv"
TEST_CSV = RESULTS_DIR / "predictions_test.csv"

print("Results source:", RESULTS_SOURCE)
print("Results dir:", RESULTS_DIR.resolve())
print("Validation CSV exists:", VAL_CSV.exists())
print("Test CSV exists:", TEST_CSV.exists())

In [ ]:
if not VAL_CSV.exists() or not TEST_CSV.exists():
    raise FileNotFoundError(
        "Run evaluate.py first for val and test splits. "
        f"Missing: {VAL_CSV if not VAL_CSV.exists() else TEST_CSV}"
    )

val_df = pd.read_csv(VAL_CSV).copy()
test_df = pd.read_csv(TEST_CSV).copy()

# Set True to overwrite legacy CSVs after direction repair.
WRITE_REPAIRED_FILES = False


# evaluate.py writes Hs/Tp in physical units and direction channels as sin/cos.
# We validate this here and normalize legacy prediction files if needed.
def _check_hs_tp_scale(name, df):
    needed = ["target_hs", "pred_hs", "target_tp", "pred_tp"]
    if not all(c in df.columns for c in needed):
        print(f"WARNING: {name} missing expected Hs/Tp columns: {needed}")
        return
    hs_vals = np.concatenate([df["target_hs"].values, df["pred_hs"].values])
    tp_vals = np.concatenate([df["target_tp"].values, df["pred_tp"].values])
    hs_vals = hs_vals[np.isfinite(hs_vals)]
    tp_vals = tp_vals[np.isfinite(tp_vals)]
    if hs_vals.size == 0 or tp_vals.size == 0:
        print(f"WARNING: {name} Hs/Tp contain no finite values")
        return
    looks_norm = (
        np.nanmin(hs_vals) >= -0.05
        and np.nanmax(hs_vals) <= 1.05
        and np.nanmin(tp_vals) >= -0.05
        and np.nanmax(tp_vals) <= 1.05
    )
    if looks_norm:
        print(
            f"WARNING: {name} Hs/Tp still look normalized. Re-run evaluate.py with the updated pipeline."
        )
    else:
        print(f"{name} Hs/Tp look un-normalized (physical units).")


def _pair_summary(df, prefix):
    p_sin_col = f"pred_{prefix}_sin"
    p_cos_col = f"pred_{prefix}_cos"
    req = [p_sin_col, p_cos_col]
    if not all(c in df.columns for c in req):
        return None
    p_sin = df[p_sin_col].to_numpy(dtype=float)
    p_cos = df[p_cos_col].to_numpy(dtype=float)
    mask = np.isfinite(p_sin) & np.isfinite(p_cos)
    if not np.any(mask):
        return None
    norms = np.hypot(p_sin[mask], p_cos[mask])
    return {
        "max_abs": float(np.nanmax(np.abs(np.concatenate([p_sin[mask], p_cos[mask]])))),
        "median_norm": float(np.nanmedian(norms)) if norms.size else np.nan,
        "count": int(mask.sum()),
    }


def _normalize_pair_in_place(df, prefix, name):
    p_sin_col = f"pred_{prefix}_sin"
    p_cos_col = f"pred_{prefix}_cos"
    t_sin_col = f"target_{prefix}_sin"
    t_cos_col = f"target_{prefix}_cos"
    req = [p_sin_col, p_cos_col, t_sin_col, t_cos_col]
    if not all(c in df.columns for c in req):
        print(f"WARNING: {name} missing {prefix} columns: {req}")
        return False

    p_sin = df[p_sin_col].to_numpy(dtype=float)
    p_cos = df[p_cos_col].to_numpy(dtype=float)
    mask = np.isfinite(p_sin) & np.isfinite(p_cos)
    if not np.any(mask):
        print(f"WARNING: {name} {prefix} has no finite prediction pairs")
        return False

    out_of_bounds = (np.nanmax(np.abs(p_sin[mask])) > 1.000001) or (
        np.nanmax(np.abs(p_cos[mask])) > 1.000001
    )
    norms = np.hypot(p_sin[mask], p_cos[mask])
    med_norm = float(np.nanmedian(norms)) if norms.size else np.nan
    needs_norm = out_of_bounds or (med_norm < 0.98) or (med_norm > 1.02)

    if needs_norm:
        safe = norms.copy()
        safe[safe == 0.0] = 1.0
        p_sin_norm = p_sin.copy()
        p_cos_norm = p_cos.copy()
        p_sin_norm[mask] = p_sin[mask] / safe
        p_cos_norm[mask] = p_cos[mask] / safe
        df[p_sin_col] = np.clip(p_sin_norm, -1.0, 1.0)
        df[p_cos_col] = np.clip(p_cos_norm, -1.0, 1.0)
        print(f"{name} {prefix}: normalized prediction vectors to unit sin/cos.")
        return True

    print(f"{name} {prefix}: predictions already valid sin/cos (median norm {med_norm:.3f}).")
    return False


for nm, csv_path, frame in [("val", VAL_CSV, val_df), ("test", TEST_CSV, test_df)]:
    _check_hs_tp_scale(nm, frame)
    changed_any = False
    for prefix in ("dir", "dp"):
        before = _pair_summary(frame, prefix)
        changed = _normalize_pair_in_place(frame, prefix, nm)
        after = _pair_summary(frame, prefix)
        changed_any = changed_any or changed
        if before and after:
            print(
                f"{nm} {prefix}: max|pred| {before['max_abs']:.3f} -> {after['max_abs']:.3f}; "
                f"median norm {before['median_norm']:.3f} -> {after['median_norm']:.3f}"
            )
    if changed_any and WRITE_REPAIRED_FILES:
        frame.to_csv(csv_path, index=False)
        print(f"{nm}: wrote repaired file to {csv_path}")
    elif changed_any:
        print(f"{nm}: repaired in memory only (set WRITE_REPAIRED_FILES=True to overwrite CSV).")
    else:
        print(f"{nm}: no direction repair needed.")

print("val rows:", len(val_df), "| test rows:", len(test_df))
val_df.head()

In [ ]:
# Confirm configured test sites match test CSV contents
import yaml

cfg_path = ROOT / "configs" / "training.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) if cfg_path.exists() else {}
configured_test_sites = [str(h) for h in (cfg.get("data", {}).get("test_sites", []) or [])]

test_sites = sorted(test_df["site"].unique())
print("Configured test_sites:", configured_test_sites)
print("Unique sites in test CSV:", test_sites)

if configured_test_sites:
    missing = sorted(set(configured_test_sites) - set(test_sites))
    extra = sorted(set(test_sites) - set(configured_test_sites))
    if not missing and not extra:
        print("\nCONFIRM: test CSV contains exactly the configured test sites.")
    else:
        print("\nWARNING: test CSV does not exactly match configured test sites.")
        if missing:
            print("Missing from test CSV:", missing)
        if extra:
            print("Extra sites in test CSV:", extra)
else:
    print("\nNo test_sites configured in configs/training.yaml")

In [ ]:
def _r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 0.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot


def _rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_pred) - np.asarray(y_true)) ** 2)))


def _bias(y_true, y_pred):
    return float(np.mean(np.asarray(y_pred) - np.asarray(y_true)))


def _pearson(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def _direction_r2(t_sin, t_cos, p_sin, p_cos):
    # Robust circular R2 computed in angle space (wrapped residuals).
    t_sin = np.asarray(t_sin, dtype=float)
    t_cos = np.asarray(t_cos, dtype=float)
    p_sin = np.asarray(p_sin, dtype=float)
    p_cos = np.asarray(p_cos, dtype=float)
    mask = np.isfinite(t_sin) & np.isfinite(t_cos) & np.isfinite(p_sin) & np.isfinite(p_cos)
    if not np.any(mask):
        return 0.0
    true_ang = np.arctan2(t_sin[mask], t_cos[mask])
    pred_ang = np.arctan2(p_sin[mask], p_cos[mask])
    delta = np.arctan2(np.sin(pred_ang - true_ang), np.cos(pred_ang - true_ang))
    ss_res = float(np.sum(delta**2))
    mu = float(np.arctan2(np.mean(np.sin(true_ang)), np.mean(np.cos(true_ang))))
    centered = np.arctan2(np.sin(true_ang - mu), np.cos(true_ang - mu))
    ss_tot = float(np.sum(centered**2))
    if ss_tot <= 0.0:
        return 0.0
    return float(1.0 - (ss_res / ss_tot))


def _direction_metrics(df, prefix="dir"):
    t_sin = df[f"target_{prefix}_sin"].values
    t_cos = df[f"target_{prefix}_cos"].values
    p_sin = df[f"pred_{prefix}_sin"].values
    p_cos = df[f"pred_{prefix}_cos"].values

    t_ang = np.arctan2(t_sin, t_cos)
    p_ang = np.arctan2(p_sin, p_cos)
    d_ang = np.arctan2(np.sin(p_ang - t_ang), np.cos(p_ang - t_ang))
    d_ang_deg = np.degrees(d_ang)

    return {
        "loss_mse": float(np.mean(d_ang**2)),
        "r2": float(_direction_r2(t_sin, t_cos, p_sin, p_cos)),
        "rmse": float(np.sqrt(np.mean(d_ang_deg**2))),
        "bias": float(np.mean(d_ang_deg)),
        "pearson": float(0.5 * (_pearson(t_sin, p_sin) + _pearson(t_cos, p_cos))),
    }


def summarize(df, split_name):
    rows = []
    for var in ["hs", "tp"]:
        t = df[f"target_{var}"].values
        p = df[f"pred_{var}"].values
        rows.append(
            {
                "split": split_name,
                "variable": var,
                "loss_mse": float(np.mean((p - t) ** 2)),
                "r2": _r2(t, p),
                "rmse": _rmse(t, p),
                "bias": _bias(t, p),
                "pearson": _pearson(t, p),
            }
        )

    d = _direction_metrics(df, prefix="dir")
    rows.append(
        {
            "split": split_name,
            "variable": "direction_deg",
            "loss_mse": d["loss_mse"],
            "r2": d["r2"],
            "rmse": d["rmse"],
            "bias": d["bias"],
            "pearson": d["pearson"],
        }
    )

    d_dp = _direction_metrics(df, prefix="dp")
    rows.append(
        {
            "split": split_name,
            "variable": "dp_deg",
            "loss_mse": d_dp["loss_mse"],
            "r2": d_dp["r2"],
            "rmse": d_dp["rmse"],
            "bias": d_dp["bias"],
            "pearson": d_dp["pearson"],
        }
    )
    return pd.DataFrame(rows)


summary = pd.concat([summarize(val_df, "val"), summarize(test_df, "test")], ignore_index=True)
summary

In [ ]:
# Taylor diagrams rebuilt using metocean_stats.plots.taylor_diagram
import numpy as np
import matplotlib.pyplot as plt
from metocean_stats import plots as mplots
from matplotlib import lines as mlines
from pathlib import Path
import pandas as pd
from IPython.display import display


# Circular helpers
def resultant_length(angles_rad):
    s = np.nanmean(np.sin(angles_rad))
    c = np.nanmean(np.cos(angles_rad))
    if np.isnan(s) or np.isnan(c):
        return np.nan
    return np.sqrt(s * s + c * c)


def circular_std(angles_rad):
    R = resultant_length(angles_rad)
    if R is None or np.isnan(R) or R <= 0:
        return np.nan
    return np.sqrt(-2.0 * np.log(R))  # in radians


def circular_correlation(ang1_rad, ang2_rad):
    ang1 = np.asarray(ang1_rad)
    ang2 = np.asarray(ang2_rad)
    mask = ~np.isnan(ang1) & ~np.isnan(ang2)
    if mask.sum() < 2:
        return np.nan
    a1 = ang1[mask]
    a2 = ang2[mask]
    mean1 = np.arctan2(np.mean(np.sin(a1)), np.mean(np.cos(a1)))
    mean2 = np.arctan2(np.mean(np.sin(a2)), np.mean(np.cos(a2)))
    num = np.sum(np.sin(a1 - mean1) * np.sin(a2 - mean2))
    den = np.sqrt(np.sum(np.sin(a1 - mean1) ** 2) * np.sum(np.sin(a2 - mean2) ** 2))
    if den == 0:
        return np.nan
    return num / den


# Save outputs under the existing test results directory (parent of TEST_CSV)
test_root = TEST_CSV.parent
taylor_out = test_root / "taylor_diagrams"
taylor_out.mkdir(parents=True, exist_ok=True)

sites = sorted(test_df["site"].unique())
cmap = plt.get_cmap("tab10")


# helper to make readable titles
def pretty_name_from_var(var_name, circular=False, angle_pairs=None):
    if circular:
        if angle_pairs and "dir" in angle_pairs[0]:
            return "Dir (circular std)"
        if angle_pairs and "dp" in angle_pairs[0]:
            return "Dp (circular std)"
        return "Direction (circular)"
    if var_name is None:
        return ""
    base = var_name.replace("target_", "").replace("pred_", "")
    short = base.split("_")[0]
    mapping = {"hs": "Hs", "tp": "Tp", "dir": "Dir", "dp": "Dp"}
    return mapping.get(short.lower(), base.replace("_", " ").title())


# helper to build a base Taylor diagram using metocean_stats and overlay per-site points
def build_and_overlay(
    var_name_ref, var_name_pred, fname, circular=False, angle_pairs=None, title=None
):
    std_refs = []
    std_models = []
    corrs = []
    for s in sites:
        sdf = test_df[test_df["site"] == s]
        if sdf.shape[0] < 2:
            std_refs.append(np.nan)
            std_models.append(np.nan)
            corrs.append(np.nan)
            continue
        if not circular:
            t = sdf[var_name_ref].values
            p = sdf[var_name_pred].values
            std_refs.append(np.nanstd(t))
            std_models.append(np.nanstd(p))
            corrs.append(np.corrcoef(t, p)[0, 1] if len(t) > 1 else np.nan)
        else:
            t_sin = sdf[angle_pairs[0]].values
            t_cos = sdf[angle_pairs[1]].values
            p_sin = sdf[angle_pairs[2]].values
            p_cos = sdf[angle_pairs[3]].values
            t_ang = np.arctan2(t_sin, t_cos)
            p_ang = np.arctan2(p_sin, p_cos)
            if len(t_ang) < 2:
                std_refs.append(np.nan)
                std_models.append(np.nan)
                corrs.append(np.nan)
                continue
            std_refs.append(circular_std(t_ang))
            std_models.append(circular_std(p_ang))
            corrs.append(circular_correlation(t_ang, p_ang))

    std_refs = np.array(std_refs, dtype=float)
    std_models = np.array(std_models, dtype=float)
    corrs = np.array(corrs, dtype=float)

    # desired maximum normalized std (use 1.0 as minimum)
    with np.errstate(divide="ignore", invalid="ignore"):
        rvals = std_models / std_refs
    max_r = np.nanmax(np.concatenate(([1.0], rvals[np.isfinite(rvals)])))
    if not np.isfinite(max_r) or max_r <= 0:
        max_r = 1.5

    # synthetic df to set plot scale
    N = 200
    base = np.linspace(0, 1, N)
    df_grid = pd.DataFrame({"grid_ref": base, "grid_comp": base * max_r})

    # create base figure with metocean_stats
    fig = mplots.taylor_diagram(df_grid, ["grid_ref"], ["grid_comp"], norm_std=True, output_file="")
    ax = fig.axes[0]

    # remove any autogenerated legend referencing grid series
    leg = ax.get_legend()
    if leg:
        leg.remove()

    # set descriptive title
    title_text = (
        title
        if title
        else pretty_name_from_var(var_name_ref, circular=circular, angle_pairs=angle_pairs)
    )
    ax.set_title(f"Taylor Diagram - {title_text}", fontsize=14)

    # overlay per-site points
    plotted_handles = []
    for i, s in enumerate(sites):
        if not np.isfinite(std_refs[i]) or not np.isfinite(std_models[i]) or std_refs[i] == 0:
            continue
        cc = corrs[i] if np.isfinite(corrs[i]) else 0.0
        r = std_models[i] / std_refs[i]
        xi = r * cc
        yi = r * np.sqrt(np.clip(1 - cc**2, 0.0, 1.0))
        color = cmap(i % 10)
        (h,) = ax.plot(xi, yi, marker="o", color=color, ms=8, linestyle="None")
        plotted_handles.append(h)
        ax.text(xi, yi, f" {s}", fontsize=8, va="bottom", ha="left")

    # create custom legend: reference + generic sites marker (sites are colored individually on the plot)
    ref_handle = mlines.Line2D(
        [], [], color="k", marker="o", linestyle="None", markersize=8, label="Reference"
    )
    sites_handle = mlines.Line2D(
        [],
        [],
        color="gray",
        marker="o",
        linestyle="None",
        markersize=8,
        label="Sites (colored by site)",
    )
    ax.legend(handles=[ref_handle, sites_handle], loc="upper right", fontsize="medium")

    out_path = taylor_out / fname
    fig.savefig(out_path, dpi=200)
    display(fig)
    print("Saved", out_path)


# Hs
build_and_overlay("target_hs", "pred_hs", "taylor_hs.png")

# Tp
build_and_overlay("target_tp", "pred_tp", "taylor_tp.png")

# Dir (circular)
build_and_overlay(
    None,
    None,
    "taylor_dir.png",
    circular=True,
    angle_pairs=("target_dir_sin", "target_dir_cos", "pred_dir_sin", "pred_dir_cos"),
)

# Dp (circular)
build_and_overlay(
    None,
    None,
    "taylor_dp.png",
    circular=True,
    angle_pairs=("target_dp_sin", "target_dp_cos", "pred_dp_sin", "pred_dp_cos"),
)

In [ ]:
# Density scatter plots (hexbin) for Dir and Dp components (sin, cos)
# Plots for Validation and for all Test points.
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

# save under test results directory
test_root = TEST_CSV.parent
density_out = test_root / "density"
density_out.mkdir(parents=True, exist_ok=True)


def plot_density_components_for_split(df, split_name, gridsize=80, cmap="viridis", out_dir=None):
    comps = [("dir", "sin"), ("dir", "cos"), ("dp", "sin"), ("dp", "cos")]
    fig, axs = plt.subplots(1, 4, figsize=(20, 4), dpi=120)
    for ax, (prefix, comp) in zip(axs, comps):
        t_col = f"target_{prefix}_{comp}"
        p_col = f"pred_{prefix}_{comp}"
        if t_col in df.columns and p_col in df.columns:
            x = df[t_col].to_numpy(dtype=float)
            y = df[p_col].to_numpy(dtype=float)
            mask = np.isfinite(x) & np.isfinite(y)
            if mask.any():
                hb = ax.hexbin(x[mask], y[mask], gridsize=gridsize, cmap=cmap, bins="log", mincnt=1)
                fig.colorbar(hb, ax=ax, label="log10(count)")
                ax.plot([-1.05, 1.05], [-1.05, 1.05], "k--", lw=1)
                ax.set_xlim(-1.05, 1.05)
                ax.set_ylim(-1.05, 1.05)
                ax.set_aspect("equal", adjustable="box")
                ax.set_xlabel(f"Target {comp}")
                ax.set_ylabel(f"Pred {comp}")
                ax.set_title(f"{prefix.upper()} {comp} ({split_name})")
            else:
                ax.text(0.5, 0.5, "No finite pairs", ha="center")
        else:
            ax.text(0.5, 0.5, f"Missing {t_col} or {p_col}", ha="center")
    plt.tight_layout()
    if out_dir is None:
        out_dir = density_out
    out_dir.mkdir(parents=True, exist_ok=True)
    safe = re.sub(r"[^A-Za-z0-9_.-]", "_", split_name)
    out_path = out_dir / f"density_{safe}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved", out_path)
    return fig, axs


# Plot for validation
plot_density_components_for_split(val_df, "Validation")

# Plot for all test rows (all holdout/test points combined)
plot_density_components_for_split(test_df, "Test_all")

In [ ]:
# Per-site density (hexbin) plots for Dir and Dp components (sin, cos)
# Generates one 1x4 figure per test/holdout site and saves PNGs to the test results directory
import re
import numpy as np
import matplotlib.pyplot as plt

# Save under the test results directory
test_root = TEST_CSV.parent
out_dir = test_root / "density_per_site"
out_dir.mkdir(parents=True, exist_ok=True)

# Determine holdout/test sites (same logic used earlier)
cfg_path = ROOT / "configs" / "training.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) if cfg_path.exists() else {}
configured_test_sites = (
    [str(h) for h in (cfg.get("data", {}).get("test_sites", []) or [])] if cfg else []
)
if not configured_test_sites:
    configured_test_sites = sorted(test_df["site"].unique())

comps = [("dir", "sin"), ("dir", "cos"), ("dp", "sin"), ("dp", "cos")]


def _safe_name(s):
    return re.sub(r"[^A-Za-z0-9_.-]", "_", str(s))


for site in configured_test_sites:
    s_df = test_df[test_df["site"] == site]
    if s_df.empty:
        print(f"No test rows for site: {site}")
        continue

    fig, axs = plt.subplots(1, 4, figsize=(20, 4), dpi=120)
    for ax, (prefix, comp) in zip(axs, comps):
        t_col = f"target_{prefix}_{comp}"
        p_col = f"pred_{prefix}_{comp}"
        if t_col not in s_df.columns or p_col not in s_df.columns:
            ax.text(0.5, 0.5, f"Missing {t_col} or {p_col}", ha="center")
            ax.set_axis_off()
            continue

        x = s_df[t_col].to_numpy(dtype=float)
        y = s_df[p_col].to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        if not mask.any():
            ax.text(0.5, 0.5, "No finite pairs", ha="center")
            ax.set_axis_off()
            continue

        hb = ax.hexbin(x[mask], y[mask], gridsize=80, cmap="viridis", bins="log", mincnt=1)
        cb = fig.colorbar(hb, ax=ax)
        cb.set_label("log10(count)")

        ax.plot([-1.05, 1.05], [-1.05, 1.05], "k--", lw=1)
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(-1.05, 1.05)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(f"Target {comp}")
        ax.set_ylabel(f"Pred {comp}")
        ax.set_title(f"{prefix.upper()} {comp}")

    plt.suptitle(f"Density components — Test site: {site}")
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    safe = _safe_name(site)
    out_path = out_dir / f"density_{safe}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight", pad_inches=0.1)
    plt.show()
    plt.close(fig)
    print(f"Saved per-site density plot: {out_path}")

In [ ]:
# Metrics tables: Validation + per-test-site
# Computes R2, MSE, RMSE, bias, Pearson for Hs, Tp, Dir, Dp
import re
from IPython.display import display, HTML
import numpy as np
import pandas as pd
from pathlib import Path

# Save under the test results directory
test_root = TEST_CSV.parent
out_dir = test_root / "metrics"
out_dir.mkdir(parents=True, exist_ok=True)


def compute_scalar_metrics(df, tcol, pcol):
    if tcol not in df.columns or pcol not in df.columns:
        return {
            "count": 0,
            "mse": np.nan,
            "r2": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "pearson": np.nan,
        }
    t = df[tcol].to_numpy(dtype=float)
    p = df[pcol].to_numpy(dtype=float)
    mask = np.isfinite(t) & np.isfinite(p)
    n = int(mask.sum())
    if n == 0:
        return {
            "count": 0,
            "mse": np.nan,
            "r2": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "pearson": np.nan,
        }
    t_ = t[mask]
    p_ = p[mask]
    mse = float(np.mean((p_ - t_) ** 2))
    r2 = float(_r2(t_, p_))
    rmse = float(_rmse(t_, p_))
    bias = float(_bias(t_, p_))
    pearson = float(_pearson(t_, p_))
    return {"count": n, "mse": mse, "r2": r2, "rmse": rmse, "bias": bias, "pearson": pearson}


def compute_all_metrics_for_df(df, label):
    rows = []
    # scalar vars Hs and Tp
    for var in ["hs", "tp"]:
        d = compute_scalar_metrics(df, f"target_{var}", f"pred_{var}")
        rows.append(
            {
                "split": label,
                "site": label,
                "variable": var,
                "count": d["count"],
                "mse": d["mse"],
                "r2": d["r2"],
                "rmse": d["rmse"],
                "bias": d["bias"],
                "pearson": d["pearson"],
            }
        )

    # circular vars: dir, dp
    for prefix, varname in [("dir", "direction_deg"), ("dp", "dp_deg")]:
        req = [
            f"target_{prefix}_sin",
            f"target_{prefix}_cos",
            f"pred_{prefix}_sin",
            f"pred_{prefix}_cos",
        ]
        if not all(c in df.columns for c in req):
            rows.append(
                {
                    "split": label,
                    "site": label,
                    "variable": varname,
                    "count": 0,
                    "mse": np.nan,
                    "r2": np.nan,
                    "rmse": np.nan,
                    "bias": np.nan,
                    "pearson": np.nan,
                }
            )
            continue
        d = _direction_metrics(df, prefix=prefix)
        mask = (
            np.isfinite(df[req[0]])
            & np.isfinite(df[req[1]])
            & np.isfinite(df[req[2]])
            & np.isfinite(df[req[3]])
        )
        n = int(mask.sum())
        rows.append(
            {
                "split": label,
                "site": label,
                "variable": varname,
                "count": n,
                "mse": float(d.get("loss_mse", np.nan)),
                "r2": float(d.get("r2", np.nan)),
                "rmse": float(d.get("rmse", np.nan)),
                "bias": float(d.get("bias", np.nan)),
                "pearson": float(d.get("pearson", np.nan)),
            }
        )

    out = pd.DataFrame(rows)[
        ["split", "site", "variable", "count", "mse", "r2", "rmse", "bias", "pearson"]
    ]
    return out


# Validation
val_table = compute_all_metrics_for_df(val_df, "Validation")
display(HTML("<h2>Validation metrics</h2>"))
display(
    val_table.style.format(
        {"mse": "{:.4f}", "r2": "{:.3f}", "rmse": "{:.3f}", "bias": "{:.3f}", "pearson": "{:.3f}"}
    )
)
val_table.to_csv(out_dir / "metrics_validation.csv", index=False)
print("Saved", out_dir / "metrics_validation.csv")

# Per-site (test) tables
test_sites = sorted(test_df["site"].unique())
for site in test_sites:
    s_df = test_df[test_df["site"] == site]
    table = compute_all_metrics_for_df(s_df, site)
    display(HTML(f"<h2>Metrics — Site: {site}</h2>"))
    display(
        table.style.format(
            {
                "mse": "{:.4f}",
                "r2": "{:.3f}",
                "rmse": "{:.3f}",
                "bias": "{:.3f}",
                "pearson": "{:.3f}",
            }
        )
    )
    safe = re.sub(r"[^A-Za-z0-9_.-]", "_", str(site))
    out_path = out_dir / f"metrics_{safe}.csv"
    table.to_csv(out_path, index=False)
    print("Saved", out_path)

# Combined
combined = pd.concat(
    [val_table]
    + [compute_all_metrics_for_df(test_df[test_df["site"] == s], s) for s in test_sites],
    ignore_index=True,
)
combined.to_csv(out_dir / "metrics_all_sites_combined.csv", index=False)
print("Saved combined", out_dir / "metrics_all_sites_combined.csv")

In [ ]:
# Scatter plots: Validation and first three test sites for Hs and Tp
import matplotlib.pyplot as plt
import numpy as np
import re
from pathlib import Path

out_dir = TEST_CSV.parent / "scatter"
out_dir.mkdir(parents=True, exist_ok=True)


def _plot_scatter_ax(ax, df, var, title=None, alpha=0.5, s=8):
    tcol = f"target_{var}"
    pcol = f"pred_{var}"
    if tcol not in df.columns or pcol not in df.columns:
        ax.text(0.5, 0.5, f"Missing {tcol} or {pcol}", ha="center")
        return
    t = df[tcol].to_numpy(dtype=float)
    p = df[pcol].to_numpy(dtype=float)
    mask = np.isfinite(t) & np.isfinite(p)
    if not mask.any():
        ax.text(0.5, 0.5, "No finite pairs", ha="center")
        return
    ax.scatter(t[mask], p[mask], alpha=alpha, s=s)
    mn = np.nanmin(np.concatenate([t[mask], p[mask]]))
    mx = np.nanmax(np.concatenate([t[mask], p[mask]]))
    pad = 0.02 * (mx - mn if mx > mn else max(abs(mx), 1.0))
    ax.plot([mn - pad, mx + pad], [mn - pad, mx + pad], "k--", lw=1)
    ax.set_xlim(mn - pad, mx + pad)
    ax.set_ylim(mn - pad, mx + pad)
    ax.set_xlabel("Target")
    ax.set_ylabel("Pred")
    try:
        d = compute_scalar_metrics(df, tcol, pcol)
        met = f"n={d['count']}, r2={d['r2']:.3f}, rmse={d['rmse']:.3f}, bias={d['bias']:.3f}"
    except Exception:
        met = ""
    if title:
        ax.set_title(f"{title}\n{met}")
    else:
        ax.set_title(met)


# Validation: Hs and Tp
fig, axs = plt.subplots(1, 2, figsize=(12, 5), dpi=120)
_plot_scatter_ax(axs[0], val_df, "hs", title="Validation - Hs")
_plot_scatter_ax(axs[1], val_df, "tp", title="Validation - Tp")
plt.tight_layout()
fpath = out_dir / "scatter_validation_hs_tp.png"
fig.savefig(fpath, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", fpath)

# Per-site: first three test sites (or configured test sites if present)
sites_to_plot = (
    configured_test_sites
    if "configured_test_sites" in globals() and configured_test_sites
    else test_sites
)[:15]
print(sites_to_plot)
for site in sites_to_plot:
    s_df = test_df[test_df["site"] == site]
    fig, axs = plt.subplots(1, 2, figsize=(12, 5), dpi=120)
    _plot_scatter_ax(axs[0], s_df, "hs", title=f"{site} - Hs")
    _plot_scatter_ax(axs[1], s_df, "tp", title=f"{site} - Tp")
    plt.tight_layout()
    safe = re.sub(r"[^A-Za-z0-9_.-]", "_", str(site))
    fpath = out_dir / f"scatter_{safe}_hs_tp.png"
    fig.savefig(fpath, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved", fpath)